# Semana 48 · ¿Y ahora qué?

El último cuaderno del plan tiene dos partes: un ejemplo completo que junta varias semanas, **la escalera de distancias de principio a fin**, y una plantilla para tu propio proyecto. **Todos los datos de la primera parte son simulados**, con valores realistas, para que puedas comparar cada resultado con la verdad.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 45 minutos la primera parte; la segunda, lo que quieras.

## Parte 1. La escalera de distancias en un cuaderno

Cuatro peldaños, cada uno calibrado con el anterior (semanas 37, 40 y 41):

1. **Paralaje** → distancia a cefeidas de la Vía Láctea.
2. **Cefeidas** → ley de Leavitt → distancia a galaxias cercanas.
3. **Supernovas de tipo Ia** en esas galaxias → su magnitud absoluta.
4. **Supernovas en galaxias lejanas** → distancias → velocidades → constante de Hubble.

### Peldaño 1: paralajes de cefeidas cercanas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(48)

# la «verdad» que el universo simulado esconde
def leavitt_real(logP):
    return -2.43 * (logP - 1) - 4.05
M_SN_real = -19.3                      # magnitud absoluta de una supernova de tipo Ia
H0_real = 70.0

# 30 cefeidas de la Vía Láctea a entre 300 y 3000 pc, con paralajes medidas por Gaia (error de 0,02 mas)
n = 30
d_real = rng.uniform(300, 3000, n)
p_med = 1000 / d_real + rng.normal(0, 0.02, n)          # mas
logP = rng.uniform(0.5, 1.6, n)
m_cef = leavitt_real(logP) + 5 * np.log10(d_real / 10) + rng.normal(0, 0.15, n)

d_med = 1000 / p_med
M_cef = m_cef - 5 * np.log10(d_med / 10)
print(f"error relativo típico de las distancias: {np.median(0.02 / p_med) * 100:.1f} %")

### Peldaño 2: calibrar la ley de Leavitt

In [ ]:
b, a = np.polyfit(logP - 1, M_cef, 1)
res = M_cef - (a + b * (logP - 1))
print(f"ley calibrada: M = {b:.2f} (log P − 1) {a:+.2f}   (verdadera: −2,43 y −4,05);  dispersión {res.std(ddof=2):.2f} mag")

plt.figure(figsize=(7, 4.5))
plt.scatter(logP, M_cef, s=15)
lp = np.linspace(0.4, 1.7, 2); plt.plot(lp, a + b * (lp - 1), "r")
plt.gca().invert_yaxis(); plt.xlabel("log P"); plt.ylabel("M"); plt.title("Peldaño 2: ley de Leavitt calibrada con paralajes")
plt.grid(alpha=0.3); plt.show()

### Peldaño 3: cefeidas y supernovas en galaxias cercanas

Ocho galaxias cercanas (entre 7 y 40 Mpc) en las que se han visto cefeidas y, además, una supernova de tipo Ia. Con las cefeidas medimos la distancia de cada galaxia; con esa distancia, la magnitud absoluta de su supernova.

In [ ]:
def leavitt_calibrada(lp):
    return a + b * (lp - 1)

n_gal = 8
D_gal = rng.uniform(7e6, 40e6, n_gal)                    # pc
M_SN_medidas = []
for D in D_gal:
    mu = 5 * np.log10(D / 10)
    lp = rng.uniform(1.0, 1.8, 20)                        # 20 cefeidas por galaxia
    m = leavitt_real(lp) + mu + rng.normal(0, 0.2, lp.size)
    mu_med = np.mean(m - leavitt_calibrada(lp))
    m_SN = M_SN_real + mu + rng.normal(0, 0.12)
    M_SN_medidas.append(m_SN - mu_med)
M_SN_medidas = np.array(M_SN_medidas)
M_SN = M_SN_medidas.mean()
print(f"magnitud absoluta de las supernovas Ia: {M_SN:.2f} ± {M_SN_medidas.std(ddof=1) / np.sqrt(n_gal):.2f}   (verdadera: {M_SN_real})")

### Peldaño 4: supernovas lejanas y la constante de Hubble

Cuarenta supernovas de tipo Ia en galaxias a entre 50 y 400 Mpc, cada una con su magnitud aparente y la velocidad de su galaxia (con velocidades peculiares de 300 km/s).

In [ ]:
n_sn = 40
D = rng.uniform(50e6, 400e6, n_sn)
v = H0_real * D / 1e6 + rng.normal(0, 300, n_sn)
m_sn = M_SN_real + 5 * np.log10(D / 10) + rng.normal(0, 0.12, n_sn)

D_med = 10 ** ((m_sn - M_SN + 5) / 5) / 1e6              # Mpc
H0 = (v * D_med).sum() / (D_med**2).sum()                  # recta por el origen (semana 41)
edad = 3.0857e19 / H0 / 3.156e7 / 1e9

plt.figure(figsize=(8, 5))
plt.scatter(D_med, v, s=15)
x = np.array([0, 420]); plt.plot(x, H0 * x, "r", label=f"H0 = {H0:.1f} km/s/Mpc")
plt.xlabel("distancia (Mpc)"); plt.ylabel("velocidad (km/s)"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Peldaño 4: diagrama de Hubble con supernovas"); plt.show()
print(f"H0 = {H0:.1f} km/s/Mpc (verdadera: {H0_real});  1/H0 = {edad:.1f} miles de millones de años")

### ¿Dónde se propaga el error?

La gracia de la escalera es que los errores de cada peldaño se arrastran a los siguientes. Repetimos todo el proceso muchas veces (Monte Carlo, semana 40) y miramos cuánto varía $H_0$ si solo cambian, por azar, las medidas de un peldaño. Para ir deprisa, empaquetamos todo en una función.

In [ ]:
def escalera(semilla, variar):
    r = np.random.default_rng(semilla)
    fijo = np.random.default_rng(1234)
    g = lambda nombre: r if nombre in variar else fijo
    # peldaños 1 y 2
    d = g("paralaje").uniform(300, 3000, 30)
    lp = g("paralaje").uniform(0.5, 1.6, 30)
    p = 1000 / d + g("paralaje").normal(0, 0.02, 30)
    m = leavitt_real(lp) + 5 * np.log10(d / 10) + g("paralaje").normal(0, 0.15, 30)
    bb, aa = np.polyfit(lp - 1, m - 5 * np.log10(1000 / p / 10), 1)
    # peldaño 3
    MS = []
    for _ in range(8):
        Dg = g("cefeidas").uniform(7e6, 40e6); mu = 5 * np.log10(Dg / 10)
        l = g("cefeidas").uniform(1.0, 1.8, 20)
        mm = leavitt_real(l) + mu + g("cefeidas").normal(0, 0.2, 20)
        MS.append(M_SN_real + mu + g("cefeidas").normal(0, 0.12) - np.mean(mm - (aa + bb * (l - 1))))
    MSN = np.mean(MS)
    # peldaño 4
    Dl = g("supernovas").uniform(50e6, 400e6, 40)
    vv = H0_real * Dl / 1e6 + g("supernovas").normal(0, 300, 40)
    ms = M_SN_real + 5 * np.log10(Dl / 10) + g("supernovas").normal(0, 0.12, 40)
    Dm = 10 ** ((ms - MSN + 5) / 5) / 1e6
    return (vv * Dm).sum() / (Dm**2).sum()

for peldano in ["paralaje", "cefeidas", "supernovas"]:
    H = np.array([escalera(s, {peldano}) for s in range(300)])
    print(f"variando solo «{peldano}»: H0 = {H.mean():.1f} ± {H.std():.1f} km/s/Mpc")
H = np.array([escalera(s, {"paralaje", "cefeidas", "supernovas"}) for s in range(300)])
print(f"variando todo:           H0 = {H.mean():.1f} ± {H.std():.1f} km/s/Mpc")

Así trabajan los equipos que miden $H_0$: buscan qué peldaño aporta más incertidumbre y tratan de mejorarlo. En el universo real, además de estos errores aleatorios, preocupan los sistemáticos (el polvo, la composición química de las cefeidas, la calibración de los instrumentos), que no se ven en esta simulación y están en el centro de la tensión de Hubble.

## Parte 2. Tu proyecto

Una plantilla con la estructura de un pequeño trabajo científico. Copia el cuaderno a tu Drive (Archivo → Guardar una copia en Drive) y rellénala con tu proyecto. Hay ideas en la página de la semana 48.

### 1. La pregunta

*Escribe aquí, en una o dos frases, qué quieres averiguar. Una buena pregunta es concreta y se puede responder con datos: «¿cuál es el periodo de δ Cephei según mis observaciones?» es mejor que «¿cómo son las estrellas variables?».*

### 2. Los datos

*¿De dónde salen? ¿Observaciones tuyas, un catálogo público, una simulación? Si son de otros, apunta la fuente y la licencia.*

In [ ]:
# Carga o genera aquí tus datos.
# Ejemplo: tus estimaciones de δ Cephei, como lista de (día juliano o fecha, magnitud)
datos = [
    # (0.0, 3.9),
    # (1.0, 4.2),
]
print(f"{len(datos)} medidas")

### 3. El método

*Explica qué vas a hacer con los datos y por qué: qué gráfica, qué ajuste, qué fórmula de qué semana.*

In [ ]:
# Tu análisis aquí

### 4. Los resultados

*Las gráficas y los números, con sus incertidumbres.*

In [ ]:
# Tus gráficas aquí

### 5. Las conclusiones y los límites

*¿Qué has encontrado? ¿Responde a la pregunta? ¿Qué podría estar mal (errores aleatorios, sistemáticos, suposiciones)? ¿Qué harías para mejorarlo?*

---

Gracias por llegar hasta aquí. El cielo seguirá ahí mañana por la noche.